# Лабораторная работа 6.1  
## REST API для ML-модели

В этом ноутбуке собран единый проект inference-сервиса для ML-модели.

Реализовано:

- обучение и сохранение демонстрационной ML-модели;
- REST API на FastAPI;
- endpoints `/health`, `/metadata`, `/predict`, `/predict_batch`;
- валидация входных данных через Pydantic;
- обработка ошибок;
- логирование запросов и latency;
- Dockerfile;
- примеры JSON-запросов;
- Python-скрипт нагрузочного тестирования;
- построение графиков latency и throughput.

Структура проекта будет создана автоматически из ячеек ноутбука.

In [ ]:
# Создаём структуру проекта

from pathlib import Path

project_dirs = [
    "app",
    "models",
    "examples",
    "load_testing",
    "results"
]

for directory in project_dirs:
    Path(directory).mkdir(parents=True, exist_ok=True)

Path("app/__init__.py").write_text("", encoding="utf-8")

print("Структура проекта создана.")

In [ ]:
%%writefile requirements.txt
fastapi==0.115.6
uvicorn[standard]==0.34.0
pydantic==2.10.4
scikit-learn==1.6.0
pandas==2.2.3
numpy==2.2.1
joblib==1.4.2
httpx==0.28.1
matplotlib==3.10.0

## 1. Обучение и сохранение модели

Для демонстрации используется синтетическая модель кредитного скоринга.

Модель решает задачу бинарной классификации:

- `0` — низкий риск дефолта;
- `1` — высокий риск дефолта.

Признаки:

- `age`;
- `income`;
- `loan_amount`;
- `employment_years`.

Модель сохраняется в файл `models/credit_scoring_model.joblib`.

Также сохраняется файл метаданных `models/metadata.json`.

In [ ]:
%%writefile train_model.py
"""
Скрипт обучения демонстрационной ML-модели.

Модель создаётся только для лабораторной работы.
В реальном проекте вместо синтетических данных следует использовать
подготовленный обучающий датасет.
"""

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score


RANDOM_STATE = 42

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "credit_scoring_model.joblib"
METADATA_PATH = MODEL_DIR / "metadata.json"


def generate_synthetic_data(n_samples: int = 5000) -> pd.DataFrame:
    """
    Генерирует синтетический датасет для задачи кредитного скоринга.

    Чем больше loan_amount относительно income и чем меньше employment_years,
    тем выше вероятность дефолта.
    """

    rng = np.random.default_rng(RANDOM_STATE)

    age = rng.integers(18, 75, size=n_samples)
    income = rng.normal(75000, 30000, size=n_samples).clip(10000, 250000)
    loan_amount = rng.normal(250000, 120000, size=n_samples).clip(10000, 1000000)
    employment_years = rng.normal(7, 5, size=n_samples).clip(0, 45)

    debt_to_income = loan_amount / income

    # Логит вероятности дефолта.
    logits = (
        -3.0
        + 1.4 * debt_to_income
        - 0.04 * employment_years
        - 0.01 * (age - 35)
        + rng.normal(0, 0.6, size=n_samples)
    )

    probability_default = 1 / (1 + np.exp(-logits))
    default = rng.binomial(1, probability_default)

    data = pd.DataFrame(
        {
            "age": age,
            "income": income,
            "loan_amount": loan_amount,
            "employment_years": employment_years,
            "default": default,
        }
    )

    return data


def main() -> None:
    features = ["age", "income", "loan_amount", "employment_years"]

    data = generate_synthetic_data()

    X = data[features]
    y = data["default"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    # Pipeline содержит preprocessing и саму модель.
    # Это важно, чтобы в inference-сервисе использовать ту же обработку данных.
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    joblib.dump(model, MODEL_PATH)

    metadata = {
        "model_name": "credit_scoring_model",
        "model_version": "1.0.0",
        "framework": "scikit-learn",
        "algorithm": "StandardScaler + LogisticRegression",
        "task_type": "binary_classification",
        "features": features,
        "input_format": "JSON object with numeric fields",
        "output": "prediction class and probability_default",
        "threshold": 0.5,
        "prepared_at": "2026-07-28",
        "metrics": {
            "accuracy": round(float(accuracy), 4),
            "roc_auc": round(float(roc_auc), 4),
        },
        "model_path": str(MODEL_PATH),
    }

    METADATA_PATH.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"Model saved to: {MODEL_PATH}")
    print(f"Metadata saved to: {METADATA_PATH}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
# Обучаем и сохраняем модель

!python train_model.py

## 2. FastAPI inference-сервис

Сервис реализует endpoints:

| Endpoint | Метод | Назначение |
|---|---|---|
| `/health` | GET | Проверка состояния сервиса |
| `/metadata` | GET | Информация о модели |
| `/predict` | POST | Предсказание для одного объекта |
| `/predict_batch` | POST | Предсказания для набора объектов |

Особенности реализации:

- модель загружается один раз при старте приложения;
- входные данные валидируются через Pydantic;
- запрещены лишние поля;
- реализована обработка ошибок;
- логируется latency каждого запроса;
- в ответе возвращается версия модели.

In [ ]:
%%writefile app/main.py
"""
FastAPI inference-сервис для ML-модели кредитного скоринга.

Запуск локально:

    uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload

После запуска документация API доступна по адресу:

    http://127.0.0.1:8000/docs
"""

import json
import logging
import os
import time
import uuid
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any

import joblib
import pandas as pd

from fastapi import FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field


# ---------------------------------------------------------------------
# Конфигурация
# ---------------------------------------------------------------------

SERVICE_NAME = "credit-scoring-inference"
SERVICE_VERSION = "1.0.0"

MODEL_PATH = Path(os.getenv("MODEL_PATH", "models/credit_scoring_model.joblib"))
METADATA_PATH = Path(os.getenv("METADATA_PATH", "models/metadata.json"))

DEFAULT_FEATURES = ["age", "income", "loan_amount", "employment_years"]

MAX_BATCH_SIZE = int(os.getenv("MAX_BATCH_SIZE", "1000"))


# ---------------------------------------------------------------------
# Логирование
# ---------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | %(levelname)s | "
        "request_id=%(request_id)s | "
        "endpoint=%(endpoint)s | "
        "status_code=%(status_code)s | "
        "latency_ms=%(latency_ms)s | "
        "model_version=%(model_version)s | "
        "%(message)s"
    ),
)

logger = logging.getLogger(SERVICE_NAME)


def log_info(
    message: str,
    request_id: str = "-",
    endpoint: str = "-",
    status_code: int | str = "-",
    latency_ms: float | str = "-",
    model_version: str = "-",
) -> None:
    """Безопасная запись информационных логов с единым набором полей."""

    logger.info(
        message,
        extra={
            "request_id": request_id,
            "endpoint": endpoint,
            "status_code": status_code,
            "latency_ms": latency_ms,
            "model_version": model_version,
        },
    )


def log_error(
    message: str,
    request_id: str = "-",
    endpoint: str = "-",
    status_code: int | str = "-",
    latency_ms: float | str = "-",
    model_version: str = "-",
) -> None:
    """Безопасная запись ошибок без передачи stack trace клиенту."""

    logger.error(
        message,
        extra={
            "request_id": request_id,
            "endpoint": endpoint,
            "status_code": status_code,
            "latency_ms": latency_ms,
            "model_version": model_version,
        },
    )


# ---------------------------------------------------------------------
# Глобальное состояние приложения
# ---------------------------------------------------------------------

class ModelState:
    """Хранилище модели и метаданных внутри процесса приложения."""

    model: Any | None = None
    metadata: dict[str, Any] = {}
    model_loaded: bool = False
    load_error: str | None = None


state = ModelState()


def load_model() -> None:
    """
    Загружает модель и метаданные.

    Важно: модель загружается один раз при старте приложения,
    а не внутри каждого HTTP-запроса.
    """

    try:
        if not MODEL_PATH.exists():
            raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

        state.model = joblib.load(MODEL_PATH)

        if METADATA_PATH.exists():
            state.metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
        else:
            state.metadata = {
                "model_name": "unknown",
                "model_version": SERVICE_VERSION,
                "features": DEFAULT_FEATURES,
                "framework": "unknown",
                "task_type": "unknown",
            }

        state.model_loaded = True
        state.load_error = None

        log_info(
            message=f"Model loaded from {MODEL_PATH}",
            model_version=state.metadata.get("model_version", SERVICE_VERSION),
        )

    except Exception as exc:
        state.model = None
        state.model_loaded = False
        state.load_error = str(exc)

        log_error(
            message=f"Model loading failed: {exc}",
            model_version=SERVICE_VERSION,
        )


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Lifecycle hook FastAPI: выполняется при старте и остановке приложения."""

    log_info(message="Application startup", model_version=SERVICE_VERSION)
    load_model()
    yield
    log_info(message="Application shutdown", model_version=SERVICE_VERSION)


app = FastAPI(
    title="Credit Scoring Inference API",
    description="REST API для получения предсказаний ML-модели.",
    version=SERVICE_VERSION,
    lifespan=lifespan,
)


# ---------------------------------------------------------------------
# Pydantic-схемы
# ---------------------------------------------------------------------

class PredictionInput(BaseModel):
    """
    Схема входных данных для одного объекта.

    extra='forbid' запрещает лишние поля.
    Например, если клиент передаст поле unknown_feature,
    сервис вернёт ошибку валидации 422.
    """

    model_config = ConfigDict(extra="forbid")

    age: int = Field(..., ge=18, le=100, description="Возраст клиента")
    income: float = Field(..., gt=0, le=10_000_000, description="Годовой доход")
    loan_amount: float = Field(..., gt=0, le=100_000_000, description="Размер кредита")
    employment_years: float = Field(..., ge=0, le=60, description="Стаж работы в годах")


class BatchPredictionInput(BaseModel):
    """Схема входных данных для batch inference."""

    model_config = ConfigDict(extra="forbid")

    items: list[PredictionInput] = Field(
        ...,
        min_length=1,
        max_length=MAX_BATCH_SIZE,
        description="Список объектов для предсказания",
    )


class PredictionOutput(BaseModel):
    """Схема ответа для одного предсказания."""

    prediction: int
    probability: float
    model_version: str


class BatchPredictionItemOutput(BaseModel):
    """Схема одного элемента ответа batch prediction."""

    prediction: int
    probability: float


class BatchPredictionOutput(BaseModel):
    """Схема ответа batch prediction."""

    predictions: list[BatchPredictionItemOutput]
    model_version: str


# ---------------------------------------------------------------------
# Middleware для логирования запросов
# ---------------------------------------------------------------------

@app.middleware("http")
async def request_logging_middleware(request: Request, call_next):
    """
    Логирует каждый HTTP-запрос.

    Фиксируются:
    - request_id;
    - endpoint;
    - status_code;
    - latency_ms;
    - model_version.
    """

    request_id = str(uuid.uuid4())
    start_time = time.perf_counter()

    request.state.request_id = request_id

    try:
        response = await call_next(request)
        status_code = response.status_code
    except Exception:
        latency_ms = round((time.perf_counter() - start_time) * 1000, 3)

        log_error(
            message="Unhandled request error",
            request_id=request_id,
            endpoint=request.url.path,
            status_code=500,
            latency_ms=latency_ms,
            model_version=state.metadata.get("model_version", SERVICE_VERSION),
        )

        raise

    latency_ms = round((time.perf_counter() - start_time) * 1000, 3)

    response.headers["X-Request-ID"] = request_id

    log_info(
        message="Request completed",
        request_id=request_id,
        endpoint=request.url.path,
        status_code=status_code,
        latency_ms=latency_ms,
        model_version=state.metadata.get("model_version", SERVICE_VERSION),
    )

    return response


# ---------------------------------------------------------------------
# Обработчики ошибок
# ---------------------------------------------------------------------

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    """Единый формат ошибок валидации."""

    request_id = getattr(request.state, "request_id", str(uuid.uuid4()))

    log_error(
        message=f"Validation error: {exc.errors()}",
        request_id=request_id,
        endpoint=request.url.path,
        status_code=422,
        model_version=state.metadata.get("model_version", SERVICE_VERSION),
    )

    return JSONResponse(
        status_code=422,
        content={
            "error": "ValidationError",
            "message": "Input data validation failed",
            "details": exc.errors(),
            "request_id": request_id,
        },
    )


@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    """Единый формат HTTP-ошибок."""

    request_id = getattr(request.state, "request_id", str(uuid.uuid4()))

    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": "HTTPException",
            "message": exc.detail,
            "request_id": request_id,
        },
    )


@app.exception_handler(Exception)
async def generic_exception_handler(request: Request, exc: Exception):
    """
    Обработка непредвиденных ошибок.

    Клиенту не возвращается traceback, так как это небезопасно.
    """

    request_id = getattr(request.state, "request_id", str(uuid.uuid4()))

    log_error(
        message=f"Internal server error: {exc}",
        request_id=request_id,
        endpoint=request.url.path,
        status_code=500,
        model_version=state.metadata.get("model_version", SERVICE_VERSION),
    )

    return JSONResponse(
        status_code=500,
        content={
            "error": "InternalServerError",
            "message": "Unexpected inference service error",
            "request_id": request_id,
        },
    )


# ---------------------------------------------------------------------
# Preprocessing, inference, postprocessing
# ---------------------------------------------------------------------

def ensure_model_loaded() -> None:
    """Проверяет, что модель доступна для inference."""

    if not state.model_loaded or state.model is None:
        raise HTTPException(
            status_code=503,
            detail=f"Model is not loaded: {state.load_error}",
        )


def preprocess(items: list[PredictionInput]) -> pd.DataFrame:
    """
    Преобразует входные Pydantic-объекты в DataFrame.

    Важно соблюдать порядок признаков, использованный при обучении модели.
    """

    features = state.metadata.get("features", DEFAULT_FEATURES)

    records = [item.model_dump() for item in items]
    dataframe = pd.DataFrame(records)

    return dataframe[features]


def run_inference(dataframe: pd.DataFrame) -> tuple[list[int], list[float]]:
    """Выполняет предсказание модели."""

    ensure_model_loaded()

    try:
        probabilities = state.model.predict_proba(dataframe)[:, 1]
        threshold = float(state.metadata.get("threshold", 0.5))
        predictions = (probabilities >= threshold).astype(int)

        return predictions.tolist(), probabilities.tolist()

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Inference failed: {exc}",
        )


def postprocess(predictions: list[int], probabilities: list[float]) -> list[dict[str, Any]]:
    """Формирует JSON-сериализуемый результат."""

    return [
        {
            "prediction": int(pred),
            "probability": round(float(prob), 6),
        }
        for pred, prob in zip(predictions, probabilities)
    ]


# ---------------------------------------------------------------------
# Endpoints
# ---------------------------------------------------------------------

@app.get("/health")
def health():
    """Проверка работоспособности сервиса."""

    if state.model_loaded:
        return {
            "status": "ok",
            "model_loaded": True,
            "service": SERVICE_NAME,
            "version": SERVICE_VERSION,
        }

    return JSONResponse(
        status_code=503,
        content={
            "status": "degraded",
            "model_loaded": False,
            "service": SERVICE_NAME,
            "version": SERVICE_VERSION,
            "error": state.load_error,
        },
    )


@app.get("/metadata")
def metadata():
    """Возвращает сведения о модели и сервисе."""

    ensure_model_loaded()

    return {
        **state.metadata,
        "service": SERVICE_NAME,
        "service_version": SERVICE_VERSION,
    }


@app.post("/predict", response_model=PredictionOutput)
def predict(payload: PredictionInput):
    """Предсказание для одного объекта."""

    start_time = time.perf_counter()

    dataframe = preprocess([payload])
    predictions, probabilities = run_inference(dataframe)
    result = postprocess(predictions, probabilities)[0]

    latency_ms = round((time.perf_counter() - start_time) * 1000, 3)

    log_info(
        message=f"Single prediction completed in {latency_ms} ms",
        endpoint="/predict",
        status_code=200,
        latency_ms=latency_ms,
        model_version=state.metadata.get("model_version", SERVICE_VERSION),
    )

    return {
        **result,
        "model_version": state.metadata.get("model_version", SERVICE_VERSION),
    }


@app.post("/predict_batch", response_model=BatchPredictionOutput)
def predict_batch(payload: BatchPredictionInput):
    """Предсказание для набора объектов."""

    start_time = time.perf_counter()

    dataframe = preprocess(payload.items)
    predictions, probabilities = run_inference(dataframe)
    results = postprocess(predictions, probabilities)

    latency_ms = round((time.perf_counter() - start_time) * 1000, 3)

    log_info(
        message=f"Batch prediction completed: batch_size={len(payload.items)}",
        endpoint="/predict_batch",
        status_code=200,
        latency_ms=latency_ms,
        model_version=state.metadata.get("model_version", SERVICE_VERSION),
    )

    return {
        "predictions": results,
        "model_version": state.metadata.get("model_version", SERVICE_VERSION),
    }

## 3. Запуск сервиса локально

Команда запуска:

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

После запуска:

- Swagger UI: `http://127.0.0.1:8000/docs`
- Health check: `http://127.0.0.1:8000/health`
- Metadata: `http://127.0.0.1:8000/metadata`

В ноутбуке следующая ячейка закомментирована, чтобы не блокировать выполнение.
При необходимости раскомментируйте команду.

In [ ]:
# Запуск FastAPI-сервиса из ноутбука.
# Эта команда блокирует ячейку, поэтому запускайте её вручную при необходимости.

# !uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload

## 4. Примеры JSON-запросов

Создадим файлы:

- `examples/request_single.json`;
- `examples/request_batch.json`.

In [ ]:
%%writefile examples/request_single.json
{
  "age": 35,
  "income": 85000,
  "loan_amount": 300000,
  "employment_years": 7
}

In [ ]:
%%writefile examples/request_batch.json
{
  "items": [
    {
      "age": 35,
      "income": 85000,
      "loan_amount": 300000,
      "employment_years": 7
    },
    {
      "age": 52,
      "income": 42000,
      "loan_amount": 150000,
      "employment_years": 3
    }
  ]
}

## 5. Примеры curl-команд

Перед выполнением команд сервис должен быть запущен.

In [ ]:
# Проверка /health

# !curl -X GET http://127.0.0.1:8000/health

In [ ]:
# Проверка /metadata

# !curl -X GET http://127.0.0.1:8000/metadata

In [ ]:
# Single prediction

# !curl -X POST http://127.0.0.1:8000/predict \
#   -H "Content-Type: application/json" \
#   -d @examples/request_single.json

In [ ]:
# Batch prediction

# !curl -X POST http://127.0.0.1:8000/predict_batch \
#   -H "Content-Type: application/json" \
#   -d @examples/request_batch.json

## 6. Docker-артефакты

Создадим:

- `Dockerfile`;
- `.dockerignore`.

Сборка образа:

```bash
docker build -t ml-inference-service .
```

Запуск контейнера:

```bash
docker run -p 8000:8000 ml-inference-service
```

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Устанавливаем системные зависимости, если они понадобятся библиотекам Python.
RUN apt-get update \
    && apt-get install -y --no-install-recommends build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --no-cache-dir --upgrade pip \
    && pip install --no-cache-dir -r requirements.txt

COPY app ./app
COPY models ./models
COPY examples ./examples

EXPOSE 8000

# Важно слушать 0.0.0.0, иначе сервис будет недоступен снаружи контейнера.
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile .dockerignore
__pycache__/
*.pyc
*.pyo
*.pyd
.ipynb_checkpoints/
.venv/
venv/
env/
.git/
.gitignore
results/
*.log
.DS_Store

## 7. Нагрузочное тестирование

Ниже создаётся Python-скрипт нагрузочного тестирования.

Он тестирует endpoint `/predict` в трёх сценариях:

| Сценарий | Виртуальные пользователи |
|---|---|
| Низкая нагрузка | 5 |
| Средняя нагрузка | 25 |
| Высокая нагрузка | 100 |

Скрипт измеряет:

- total requests;
- throughput / RPS;
- average latency;
- median latency;
- p95;
- p99;
- max latency;
- error rate.

Перед запуском скрипта сервис должен быть запущен.

In [ ]:
%%writefile load_testing/load_test.py
"""
Простой скрипт нагрузочного тестирования FastAPI inference-сервиса.

Запуск:

    python load_testing/load_test.py

По умолчанию тестируется:

    http://127.0.0.1:8000/predict

Результаты сохраняются в:

    results/load_test_results.csv
"""

import asyncio
import csv
import statistics
import time
from pathlib import Path
from typing import Any

import httpx


BASE_URL = "http://127.0.0.1:8000"
ENDPOINT = "/predict"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "load_test_results.csv"

PAYLOAD = {
    "age": 35,
    "income": 85000,
    "loan_amount": 300000,
    "employment_years": 7,
}


SCENARIOS = [
    {
        "name": "Низкая нагрузка",
        "virtual_users": 5,
        "duration_seconds": 30,
    },
    {
        "name": "Средняя нагрузка",
        "virtual_users": 25,
        "duration_seconds": 30,
    },
    {
        "name": "Высокая нагрузка",
        "virtual_users": 100,
        "duration_seconds": 30,
    },
]


def percentile(values: list[float], p: float) -> float:
    """Вычисляет перцентиль для списка значений."""

    if not values:
        return 0.0

    sorted_values = sorted(values)
    index = int((len(sorted_values) - 1) * p)
    return sorted_values[index]


async def worker(
    client: httpx.AsyncClient,
    end_time: float,
    latencies_ms: list[float],
    status_codes: list[int],
    errors: list[str],
) -> None:
    """Один виртуальный пользователь, который отправляет запросы до конца сценария."""

    while time.perf_counter() < end_time:
        start = time.perf_counter()

        try:
            response = await client.post(ENDPOINT, json=PAYLOAD)
            latency_ms = (time.perf_counter() - start) * 1000

            latencies_ms.append(latency_ms)
            status_codes.append(response.status_code)

            if response.status_code >= 400:
                errors.append(f"HTTP {response.status_code}: {response.text[:200]}")

        except Exception as exc:
            latency_ms = (time.perf_counter() - start) * 1000

            latencies_ms.append(latency_ms)
            status_codes.append(0)
            errors.append(str(exc))


async def run_scenario(scenario: dict[str, Any]) -> dict[str, Any]:
    """Запускает один сценарий нагрузки."""

    name = scenario["name"]
    virtual_users = scenario["virtual_users"]
    duration_seconds = scenario["duration_seconds"]

    print(f"Running scenario: {name}, VUs={virtual_users}, duration={duration_seconds}s")

    latencies_ms: list[float] = []
    status_codes: list[int] = []
    errors: list[str] = []

    timeout = httpx.Timeout(10.0)

    started_at = time.perf_counter()
    end_time = started_at + duration_seconds

    limits = httpx.Limits(
        max_connections=virtual_users * 2,
        max_keepalive_connections=virtual_users * 2,
    )

    async with httpx.AsyncClient(
        base_url=BASE_URL,
        timeout=timeout,
        limits=limits,
    ) as client:
        tasks = [
            worker(client, end_time, latencies_ms, status_codes, errors)
            for _ in range(virtual_users)
        ]
        await asyncio.gather(*tasks)

    finished_at = time.perf_counter()
    actual_duration = finished_at - started_at

    total_requests = len(status_codes)
    failed_requests = sum(1 for code in status_codes if code >= 400 or code == 0)
    successful_requests = total_requests - failed_requests

    throughput = total_requests / actual_duration if actual_duration > 0 else 0
    error_rate = failed_requests / total_requests if total_requests > 0 else 0

    result = {
        "scenario": name,
        "virtual_users": virtual_users,
        "duration_seconds": round(actual_duration, 3),
        "requests": total_requests,
        "successful_requests": successful_requests,
        "failed_requests": failed_requests,
        "avg_latency_ms": round(statistics.mean(latencies_ms), 3) if latencies_ms else 0,
        "median_latency_ms": round(statistics.median(latencies_ms), 3) if latencies_ms else 0,
        "p95_latency_ms": round(percentile(latencies_ms, 0.95), 3),
        "p99_latency_ms": round(percentile(latencies_ms, 0.99), 3),
        "max_latency_ms": round(max(latencies_ms), 3) if latencies_ms else 0,
        "throughput_rps": round(throughput, 3),
        "error_rate": round(error_rate, 6),
    }

    print(result)

    if errors:
        print(f"Errors sample: {errors[:3]}")

    return result


async def main() -> None:
    """Запускает все сценарии и сохраняет таблицу результатов."""

    results = []

    for scenario in SCENARIOS:
        result = await run_scenario(scenario)
        results.append(result)

        # Пауза между сценариями, чтобы сервис немного восстановился.
        await asyncio.sleep(3)

    fieldnames = list(results[0].keys())

    with RESULTS_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print(f"Results saved to: {RESULTS_PATH}")


if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# Запуск нагрузочного тестирования.
# Перед выполнением убедитесь, что FastAPI-сервис запущен.

# !python load_testing/load_test.py

## 8. Построение графиков по результатам нагрузочного тестирования

Скрипт строит два графика:

1. latency от числа виртуальных пользователей;
2. throughput от числа виртуальных пользователей.

Результаты сохраняются в папку `results`.

In [ ]:
%%writefile load_testing/plot_results.py
"""
Построение графиков по результатам нагрузочного тестирования.

Запуск:

    python load_testing/plot_results.py
"""

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


RESULTS_PATH = Path("results/load_test_results.csv")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)


def main() -> None:
    if not RESULTS_PATH.exists():
        raise FileNotFoundError(
            "Файл results/load_test_results.csv не найден. "
            "Сначала запустите load_testing/load_test.py"
        )

    df = pd.read_csv(RESULTS_PATH)

    print(df)

    # График latency.
    plt.figure(figsize=(8, 5))
    plt.plot(df["virtual_users"], df["avg_latency_ms"], marker="o", label="Avg latency")
    plt.plot(df["virtual_users"], df["p95_latency_ms"], marker="o", label="p95 latency")
    plt.plot(df["virtual_users"], df["p99_latency_ms"], marker="o", label="p99 latency")
    plt.xlabel("Virtual users")
    plt.ylabel("Latency, ms")
    plt.title("Latency vs Load")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "latency_vs_load.png", dpi=150)

    # График throughput.
    plt.figure(figsize=(8, 5))
    plt.plot(df["virtual_users"], df["throughput_rps"], marker="o")
    plt.xlabel("Virtual users")
    plt.ylabel("Throughput, RPS")
    plt.title("Throughput vs Load")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "throughput_vs_load.png", dpi=150)

    print("Graphs saved to results/latency_vs_load.png and results/throughput_vs_load.png")


if __name__ == "__main__":
    main()

In [ ]:
# Построение графиков.
# Выполнять после load_testing/load_test.py.

# !python load_testing/plot_results.py

## 9. README.md

Создадим README с инструкциями по запуску проекта.

In [ ]:
%%writefile README.md
# Лабораторная работа 6.1  
## REST API для ML-модели

Проект реализует inference-сервис для демонстрационной ML-модели кредитного скоринга.

## Стек

- Python
- FastAPI
- Pydantic
- scikit-learn
- Uvicorn
- Docker
- httpx для нагрузочного тестирования

## Структура проекта

```text
.
├── app/
│   ├── __init__.py
│   └── main.py
├── examples/
│   ├── request_single.json
│   └── request_batch.json
├── load_testing/
│   ├── load_test.py
│   └── plot_results.py
├── models/
│   ├── credit_scoring_model.joblib
│   └── metadata.json
├── results/
├── train_model.py
├── requirements.txt
├── Dockerfile
├── .dockerignore
└── README.md
```

## Локальный запуск

### 1. Установка зависимостей

```bash
pip install -r requirements.txt
```

### 2. Обучение модели

```bash
python train_model.py
```

### 3. Запуск сервиса

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

Swagger UI:

```text
http://127.0.0.1:8000/docs
```

## API

### GET /health

Проверка состояния сервиса.

Пример ответа:

```json
{
  "status": "ok",
  "model_loaded": true,
  "service": "credit-scoring-inference",
  "version": "1.0.0"
}
```

### GET /metadata

Информация о модели.

### POST /predict

Пример запроса:

```json
{
  "age": 35,
  "income": 85000,
  "loan_amount": 300000,
  "employment_years": 7
}
```

Пример ответа:

```json
{
  "prediction": 0,
  "probability": 0.18,
  "model_version": "1.0.0"
}
```

### POST /predict_batch

Пример запроса:

```json
{
  "items": [
    {
      "age": 35,
      "income": 85000,
      "loan_amount": 300000,
      "employment_years": 7
    },
    {
      "age": 52,
      "income": 42000,
      "loan_amount": 150000,
      "employment_years": 3
    }
  ]
}
```

## Примеры curl

```bash
curl -X GET http://127.0.0.1:8000/health
```

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d @examples/request_single.json
```

```bash
curl -X POST http://127.0.0.1:8000/predict_batch \
  -H "Content-Type: application/json" \
  -d @examples/request_batch.json
```

## Docker

### Сборка

```bash
docker build -t ml-inference-service .
```

### Запуск

```bash
docker run -p 8000:8000 ml-inference-service
```

## Нагрузочное тестирование

Перед тестом запустить сервис.

```bash
python load_testing/load_test.py
```

Построить графики:

```bash
python load_testing/plot_results.py
```

Результаты сохраняются в папку `results`.

## Известные ограничения

- используется демонстрационная синтетическая модель;
- нет production-мониторинга drift/quality;
- нет авторизации;
- нет ограничения размера HTTP body на уровне reverse proxy;
- нет горизонтального масштабирования;
- нет CI/CD.

## 10. Шаблон отчёта REPORT.md

Фактические результаты нагрузочного тестирования нужно вписать после запуска скриптов.

In [ ]:
%%writefile REPORT.md
# Отчёт по лабораторной работе 6.1  
## REST API для ML-модели

## Титульная часть

- Лабораторная работа: 6.1 REST API для модели
- ФИО студента: `<указать>`
- Группа: `<указать>`
- Дата выполнения: `<указать>`
- Фреймворк: FastAPI
- Модель: демонстрационная модель кредитного скоринга

## Описание модели

Используется модель бинарной классификации.

- Тип задачи: binary classification
- Алгоритм: StandardScaler + LogisticRegression
- Формат модели: joblib
- Файл модели: `models/credit_scoring_model.joblib`
- Версия модели: 1.0.0

Признаки:

- `age`
- `income`
- `loan_amount`
- `employment_years`

Выход:

- `prediction`: класс 0 или 1
- `probability`: вероятность дефолта

## Архитектура inference-сервиса

Схема обработки запроса:

```text
Client -> HTTP API -> Validation -> Preprocessing -> Model Inference -> Postprocessing -> Response
```

Модель загружается один раз при старте приложения через lifespan hook FastAPI.

Основные компоненты:

- FastAPI-приложение: `app/main.py`
- Валидация: Pydantic-схемы
- Preprocessing: преобразование JSON в pandas DataFrame с фиксированным порядком признаков
- Inference: вызов `model.predict_proba`
- Postprocessing: округление вероятности и формирование JSON
- Логирование: middleware FastAPI

## Описание API

### GET /health

Проверяет состояние сервиса.

Коды ответа:

- 200 — сервис работает, модель загружена
- 503 — модель не загружена

### GET /metadata

Возвращает метаданные модели.

Коды ответа:

- 200 — успешно
- 503 — модель не загружена

### POST /predict

Получает предсказание для одного объекта.

Коды ответа:

- 200 — успешно
- 422 — ошибка валидации
- 503 — модель не загружена
- 500 — ошибка inference

### POST /predict_batch

Получает предсказания для списка объектов.

Коды ответа:

- 200 — успешно
- 422 — ошибка валидации
- 503 — модель не загружена
- 500 — ошибка inference

## Контейнеризация

Dockerfile выполняет:

1. Использование базового образа `python:3.11-slim`
2. Установку зависимостей из `requirements.txt`
3. Копирование приложения, модели и примеров
4. Запуск сервиса через uvicorn

Сборка:

```bash
docker build -t ml-inference-service .
```

Запуск:

```bash
docker run -p 8000:8000 ml-inference-service
```

## Нагрузочное тестирование

Инструмент: собственный Python-скрипт на `httpx`.

Сценарии:

| Сценарий | VUs | Длительность |
|---|---:|---:|
| Низкая нагрузка | 5 | 30 сек |
| Средняя нагрузка | 25 | 30 сек |
| Высокая нагрузка | 100 | 30 сек |

Тестировался endpoint `/predict`.

## Результаты нагрузочного тестирования

После запуска `python load_testing/load_test.py` необходимо вставить сюда таблицу из `results/load_test_results.csv`.

| Сценарий | VUs | Requests | Avg latency | p95 | p99 | Max | Throughput | Error rate |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Низкая нагрузка | 5 | ... | ... | ... | ... | ... | ... | ... |
| Средняя нагрузка | 25 | ... | ... | ... | ... | ... | ... | ... |
| Высокая нагрузка | 100 | ... | ... | ... | ... | ... | ... | ... |

## Анализ результатов

Необходимо описать:

- как менялась latency при росте нагрузки;
- как менялся throughput;
- при какой нагрузке началась деградация;
- какой процент ошибок наблюдался;
- какие bottlenecks обнаружены.

Возможные bottlenecks:

- ограниченное число workers;
- CPU-bound inference;
- накладные расходы pandas preprocessing;
- высокая конкуренция запросов;
- ограничения CPU/RAM;
- отсутствие batch inference на клиентской стороне.

## Возможные оптимизации

- загружать модель один раз при старте приложения;
- использовать batch inference;
- увеличить количество uvicorn/gunicorn workers;
- оптимизировать preprocessing;
- использовать ONNX Runtime;
- уменьшить размер модели;
- добавить кэширование для повторяющихся запросов;
- использовать горизонтальное масштабирование;
- добавить reverse proxy с лимитами размера запроса;
- добавить production-мониторинг latency, error rate, drift.

## Выводы

В работе реализован REST API inference-сервис для ML-модели.

Реализованы:

- endpoints `/health`, `/metadata`, `/predict`, `/predict_batch`;
- валидация входных данных;
- обработка ошибок;
- логирование;
- контейнеризация;
- нагрузочное тестирование.

Для production deployment требуется доработать:

- мониторинг качества модели;
- авторизацию;
- CI/CD;
- версионирование моделей;
- rollback;
- горизонтальное масштабирование;
- контроль data drift.

## 11. Итог

Ноутбук создал полный набор артефактов лабораторной работы:

- код сервиса;
- код обучения модели;
- модельный артефакт;
- requirements;
- Dockerfile;
- примеры запросов;
- нагрузочное тестирование;
- README;
- REPORT.

Дальнейшие шаги:

1. Запустить сервис.
2. Проверить endpoints.
3. Провести нагрузочное тестирование.
4. Построить графики.
5. Заполнить отчёт фактическими результатами.